## Dataset creation 

In [2]:
import torch

In [ ]:
import json

with open ('ioi_dataset_100.json', 'rb') as f:
    data = json.load(f)

data[:10]

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-270m")

/home/muskaan06/miniconda3/envs/jupyter/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 236/236 [00:00<00:00, 1084.94it/s, Materializing param=model.norm.weight]                                


In [6]:
model = model.to(device)

In [7]:
# tokenized_data = []
# for eg in data:
#     temp = {}
#     temp['clean_prompt'] = tokenizer(eg['clean_prompt'], return_tensors='pt').to(device = device)
#     temp['corrupted_prompt'] = tokenizer(eg['corrupted_prompt'], return_tensors='pt').to(device = device)
#     temp['nameA'] = tokenizer("▁"+eg["name_A"], return_tensors='pt').to(device = device)
#     temp['nameB'] = tokenizer("▁"+eg["name_B"], return_tensors='pt').to(device = device)
#     tokenized_data.append(temp)

# len(tokenized_data)

In [8]:
#Then Catherine and Thomas went to the movies, and Thomas gave a ticket to

In [9]:
tokenizer.convert_tokens_to_ids('▁Thomas')

9509

In [10]:
tokenizer.convert_ids_to_tokens(36266)

'▁Catherine'

In [11]:
# tokenized_data[:10]

In [12]:
# filtered_data = []
# for example in data:
#     # print(example)
#     if len(example['clean_prompt']['input_ids']) == len(example['corrupted_prompt']['input_ids']):
#         filtered_data.append(example)

In [13]:
# len(filtered_data)

In [14]:
model.model.layers[0].self_attn

Gemma3Attention(
  (q_proj): Linear(in_features=640, out_features=1024, bias=False)
  (k_proj): Linear(in_features=640, out_features=256, bias=False)
  (v_proj): Linear(in_features=640, out_features=256, bias=False)
  (o_proj): Linear(in_features=1024, out_features=640, bias=False)
  (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
  (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
)

## Verify the IOI capability
Find logit of last token of every example -->  find logit of {B} of every example --> difference of mean of both

In [15]:
# tokenized_data = []
# for eg in data:
#     temp = {}
#     temp['clean_prompt'] = tokenizer(eg['clean_prompt'], return_tensors='pt').to(device = device)
#     temp['corrupted_prompt'] = tokenizer(eg['corrupted_prompt'], return_tensors='pt').to(device = device)
#     temp['nameA'] = tokenizer("▁"+eg["name_A"], return_tensors='pt').to(device = device)
#     temp['nameB'] = tokenizer("▁"+eg["name_B"], return_tensors='pt').to(device = device)
#     tokenized_data.append(temp)

# len(tokenized_data)

In [16]:
diff_logit = []
for example in data:
    tokenize = tokenizer(example['clean_prompt'], return_tensors='pt').to(device = device)
    with torch.no_grad():
        output = model(**tokenize)
    A_token = tokenizer("▁"+example["name_A"], return_tensors='pt').to(device = device)
    B_token = tokenizer("▁"+example["name_B"], return_tensors='pt').to(device = device)
    A_id = A_token["input_ids"][0][-1] #extracting token id of A
    B_id = B_token["input_ids"][0][-1]
    # print(A_id,"   ",B_id)
    A_logit = output.logits[:,-1,:][0][int(A_id)] #taking out logit of A from the last token's logit
    # print(A_logit)
    B_logit = output.logits[:,-1,:][0][int(B_id)]
    diff_logit.append((A_logit - B_logit).detach().cpu())
    

In [17]:
clean_prompt_mean = torch.mean(torch.stack(diff_logit))

In [18]:
torch.mean(torch.stack(diff_logit))

tensor(5.0625, dtype=torch.bfloat16)

In [19]:
data[0]

{'clean_prompt': 'Then Catherine and Thomas went to the movies, and Thomas gave a ticket to',
 'corrupted_prompt': 'Then Catherine and Thomas went to the movies, and Sarah gave a ticket to',
 'name_A': 'Catherine',
 'name_B': 'Thomas',
 'name_C': 'Sarah',
 'template': 'Then {A} and {B} went to the movies, and {B} gave a ticket to',
 'template_idx': 0}

In [20]:
#CORRUPT

diff_logit = []
for example in data:
    tokenize = tokenizer(example['corrupted_prompt'], return_tensors='pt').to(device = device)
    with torch.no_grad():
        output = model(**tokenize)
    A_token = tokenizer("▁"+example["name_A"], return_tensors='pt').to(device = device)
    B_token = tokenizer("▁"+example["name_B"], return_tensors='pt').to(device = device)
    A_id = A_token["input_ids"][0][-1] #extracting token id of A
    B_id = B_token["input_ids"][0][-1]
    # print(A_id,"   ",B_id)
    A_logit = output.logits[:,-1,:][0][int(A_id)] #taking out logit of A from the last token's logit
    # print(A_logit)
    B_logit = output.logits[:,-1,:][0][int(B_id)]
    diff_logit.append((A_logit - B_logit).detach().cpu())
    

In [ ]:
corrupt_prompt_mean = torch.mean(torch.stack(diff_logit))

tensor(0.8281, dtype=torch.bfloat16)

## Activation Patching
Take individual components --> replace with corrupt --> run model --> get final token logit --> if diff large then important

### storing activations of corrupt examples 

In [21]:
for name, layer in model.named_modules():
    print(name)


model
model.embed_tokens
model.layers
model.layers.0
model.layers.0.self_attn
model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.self_attn.q_norm
model.layers.0.self_attn.k_norm
model.layers.0.mlp
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.0.mlp.act_fn
model.layers.0.input_layernorm
model.layers.0.post_attention_layernorm
model.layers.0.pre_feedforward_layernorm
model.layers.0.post_feedforward_layernorm
model.layers.1
model.layers.1.self_attn
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.self_attn.q_norm
model.layers.1.self_attn.k_norm
model.layers.1.mlp
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.layers.1.mlp.act_fn
model.layers.1.input_layernorm
model.layers.1.post_attention_layernorm
model.layers

In [ ]:
#mlp hooks
corrupt_activations = {}
def mlp_hook(name):
    def hook(model, input, output):
        # print(output)
        if name not in corrupt_activations:
            corrupt_activations[name]=[]
        if isinstance(output, tuple):
            corrupt_activations[name].append(output[0].detach().cpu().tolist())
        else:
            corrupt_activations[name].append(output.detach().cpu().tolist())
        return output
    return hook

In [ ]:
def attn_hook(name):
    def hook(model, input, output):
        # print(output)
        for j in range(4):
            if name not in corrupt_activations:
                corrupt_activations[name]={}
            if j not in corrupt_activations[name]:
                corrupt_activations[name][j] = []
            corrupt_activations[name][j].append(input[0][:,:,j*256:(j+1)*256].detach().cpu().tolist()) #batch, token, embedding
        return output
    return hook

In [24]:
#name=layer_0 ---> 4 attn heads
#{layer_0:[{0:[],1:[],2:[]},
# {}]}

In [25]:
h = []
for example in data:
   
    for i in range(18):
        for name, layer in  model.named_modules():
            if name==f'model.layers.{i}.mlp':
              h.append(layer.register_forward_hook(mlp_hook(name)))
                
            if name==f'model.layers.{i}.self_attn.o_proj':
                # for j in range(4):
                  h.append(layer.register_forward_hook(attn_hook(name)))
            
    tokenize = tokenizer(example['corrupted_prompt'], return_tensors='pt').to(device = device)
    with torch.no_grad():
      output = model(**tokenize)
    for i in h:
      i.remove()

    # A_id = example["nameA"]["input_ids"][0][-1] #extracting token id of A
    # B_id = example["nameB"]["input_ids"][0][-1]
    # # print(A_id,"   ",B_id)
    # A_logit = output.logits[:,-1,:][0][int(A_id)] #taking out logit of A from the last token's logit
    # # print(A_logit)
    # B_logit = output.logits[:,-1,:][0][int(B_id)]
    # diff_logit.append(A_logit - B_logit)

In [26]:
#clearing registered hook
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [27]:
with open("corrput_activation.json", "w") as f:
    json.dump(corrupt_activations, f)

In [ ]:
corrupt_activations['model.layers.0.mlp']

In [ ]:
corrupt_activations['model.layers.0.self_attn.o_proj'] #layer, attention head, example indexes, activations

### patching

In [22]:
import json

with open ('corrput_activation.json', 'rb') as f:
    corrupt_activations = json.load(f)

In [23]:
len(corrupt_activations['model.layers.0.mlp'][0])

1

In [ ]:
corrupt_activations['model.layers.0.self_attn.o_proj']['0'][0]

In [25]:
#clearing registered hook
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [26]:
#mlp hooks

def mlp_hook(name, corrupt):
    def hook(model, input, output): 
        output = corrupt.to(device=device)
        # if name not in corrupt_activations:
        #     corrupt_activations[name]=[]
        # if isinstance(output, tuple):
        #     corrupt_activations[name].append(output[0].detach().cpu())
        # else:
        #     corrupt_activations[name].append(output.detach().cpu())
        return output
    return hook

In [27]:
h = []
mlp_corrupt_logits = {}
for e, example in enumerate(data):
    for i in range(18):
        for name, layer in  model.named_modules():
            if name==f'model.layers.{i}.mlp':
              # print(torch.tensor(corrupt_activations[name][e], dtype=torch.bfloat16, device=device))
              h.append(layer.register_forward_hook(mlp_hook(name, torch.tensor(corrupt_activations[name][e], dtype=torch.bfloat16, device=device))))
              tokenize = tokenizer(example['clean_prompt'], return_tensors='pt').to(device = device)
              with torch.no_grad():
                output = model(**tokenize)
              A_token = tokenizer("▁"+example["name_A"], return_tensors='pt').to(device = device)
              B_token = tokenizer("▁"+example["name_B"], return_tensors='pt').to(device = device)
              A_id = A_token["input_ids"][0][-1] #extracting token id of A
              B_id = B_token["input_ids"][0][-1]
              A_logit = output.logits[:,-1,:][0][int(A_id)] #taking out logit of A from the last token's logit
              B_logit = output.logits[:,-1,:][0][int(B_id)]
              if e not in mlp_corrupt_logits:
                 mlp_corrupt_logits[e] = []
              # print("A",A_logit)
              # print("B",B_logit)
              mlp_corrupt_logits[e].append((A_logit - B_logit).detach().cpu())
              for k in h:
                k.remove()
              # torch.cuda.empty_cache()
              # gc.collect()

In [34]:
import gc
torch.cuda.empty_cache()
gc.collect()

26230

In [35]:
len(mlp_corrupt_logits)

100

In [ ]:
mlp_corrupt_logits

In [44]:
#clearing registered hook
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [ ]:
def attn_hook(name, corrupt, j):
    def hook(module, input, output):
        patched_input = input[0].clone()
        patched_input[:, :, j*256:(j+1)*256] = corrupt
        with torch.no_grad():
            patched_output = torch.nn.functional.linear(
                patched_input, module.weight, module.bias
            )
        return patched_output
    return hook 

In [ ]:
h = []
attn_corrupt_logits = {}
for e, example in enumerate(data):
    for i in range(18):
        for name, layer in  model.named_modules():
                if name==f'model.layers.{i}.self_attn.o_proj':
                    for j in range(4):
                        c = torch.tensor(corrupt_activations[name][str(j)][e], dtype=torch.bfloat16,device=device)
                        # print(c)
                        h.append(layer.register_forward_hook(attn_hook(name, c, j)))
                        tokenize = tokenizer(example['clean_prompt'], return_tensors='pt').to(device = device)
                        with torch.no_grad():
                            output = model(**tokenize)
                        A_token = tokenizer("▁"+example["name_A"], return_tensors='pt').to(device = device)
                        B_token = tokenizer("▁"+example["name_B"], return_tensors='pt').to(device = device)
                        A_id = A_token["input_ids"][0][-1]
                        B_id = B_token["input_ids"][0][-1]
                        A_logit = output.logits[:,-1,:][0][int(A_id)] #taking out logit of A from the last token's logit
                        B_logit = output.logits[:,-1,:][0][int(B_id)]
                        if e not in attn_corrupt_logits:
                            attn_corrupt_logits[e] = {}
                        if i not in attn_corrupt_logits[e]:
                            attn_corrupt_logits[e][i] = []
                        attn_corrupt_logits[e][i].append((A_logit - B_logit).detach().cpu())
                        for k in h:
                            k.remove()

In [47]:
len(attn_corrupt_logits)

100

In [ ]:
attn_corrupt_logits

## Finding subgraph

### finding mlp component

In [ ]:
mlp_corrupt_logits

In [50]:
mlp_corrupt_logits[0] #examples, layers, activations

[tensor(4., dtype=torch.bfloat16),
 tensor(4.1250, dtype=torch.bfloat16),
 tensor(4.1250, dtype=torch.bfloat16),
 tensor(-0.1250, dtype=torch.bfloat16),
 tensor(4., dtype=torch.bfloat16),
 tensor(4.1250, dtype=torch.bfloat16),
 tensor(3.6250, dtype=torch.bfloat16),
 tensor(2.8750, dtype=torch.bfloat16),
 tensor(3.8750, dtype=torch.bfloat16),
 tensor(3.7500, dtype=torch.bfloat16),
 tensor(3.3750, dtype=torch.bfloat16),
 tensor(3.3750, dtype=torch.bfloat16),
 tensor(3.3750, dtype=torch.bfloat16),
 tensor(4.2500, dtype=torch.bfloat16),
 tensor(4.2500, dtype=torch.bfloat16),
 tensor(4.1250, dtype=torch.bfloat16),
 tensor(4.6250, dtype=torch.bfloat16),
 tensor(4.2500, dtype=torch.bfloat16)]

In [51]:
# Stack into tensor of shape (100, 18)
all_diffs = torch.stack([
    torch.stack(layer_diffs)
    for layer_diffs in mlp_corrupt_logits.values()
])  # 

print(all_diffs.shape)

torch.Size([100, 18])


In [52]:
mean_per_layer = all_diffs.mean(dim=0)  #finding mean across layer for all examples
print(mean_per_layer)
for i, mean in enumerate(mean_per_layer):
    print(f"model.layers.{i}.mlp: {mean:.4f}")

tensor([4.8438, 4.8750, 5.0625, 0.7500, 5.1250, 5.0312, 4.4062, 4.5938, 4.6875,
        4.2188, 3.9844, 4.1562, 4.6250, 5.0938, 5.0625, 5.0000, 5.0625, 5.0625],
       dtype=torch.bfloat16)
model.layers.0.mlp: 4.8438
model.layers.1.mlp: 4.8750
model.layers.2.mlp: 5.0625
model.layers.3.mlp: 0.7500
model.layers.4.mlp: 5.1250
model.layers.5.mlp: 5.0312
model.layers.6.mlp: 4.4062
model.layers.7.mlp: 4.5938
model.layers.8.mlp: 4.6875
model.layers.9.mlp: 4.2188
model.layers.10.mlp: 3.9844
model.layers.11.mlp: 4.1562
model.layers.12.mlp: 4.6250
model.layers.13.mlp: 5.0938
model.layers.14.mlp: 5.0625
model.layers.15.mlp: 5.0000
model.layers.16.mlp: 5.0625
model.layers.17.mlp: 5.0625


In [53]:
mean_mlp_diff = clean_prompt_mean - mean_per_layer  #to see the deviation from clean prompt
mean_mlp_diff

tensor([ 0.2188,  0.1875,  0.0000,  4.3125, -0.0625,  0.0312,  0.6562,  0.4688,
         0.3750,  0.8438,  1.0781,  0.9062,  0.4375, -0.0312,  0.0000,  0.0625,
         0.0000,  0.0000], dtype=torch.bfloat16)

In [54]:
ranked = sorted(enumerate(mean_mlp_diff.tolist()), key=lambda x: x[1], reverse=True)    #sorting as per the deviation, more the deviation ---> more important the component
print(ranked)
print("\nRanked layers:")
for layer_idx, score in ranked:
    print(f"layer {layer_idx}: {score:.4f}")

[(3, 4.3125), (10, 1.078125), (11, 0.90625), (9, 0.84375), (6, 0.65625), (7, 0.46875), (12, 0.4375), (8, 0.375), (0, 0.21875), (1, 0.1875), (15, 0.0625), (5, 0.03125), (2, 0.0), (14, 0.0), (16, 0.0), (17, 0.0), (13, -0.03125), (4, -0.0625)]

Ranked layers:
layer 3: 4.3125
layer 10: 1.0781
layer 11: 0.9062
layer 9: 0.8438
layer 6: 0.6562
layer 7: 0.4688
layer 12: 0.4375
layer 8: 0.3750
layer 0: 0.2188
layer 1: 0.1875
layer 15: 0.0625
layer 5: 0.0312
layer 2: 0.0000
layer 14: 0.0000
layer 16: 0.0000
layer 17: 0.0000
layer 13: -0.0312
layer 4: -0.0625


In [55]:
threshold = 0.3

In [56]:
selected_mlp_components = [(layer, r) for layer, r in ranked if r>threshold]
selected_mlp_components

[(3, 4.3125),
 (10, 1.078125),
 (11, 0.90625),
 (9, 0.84375),
 (6, 0.65625),
 (7, 0.46875),
 (12, 0.4375),
 (8, 0.375)]

### finding attention head component

In [ ]:
attn_corrupt_logits

In [63]:
all_attn_diffs = torch.stack([
    torch.stack([
        torch.stack(head_diffs)          # shape: (4,)
        for layer_idx, head_diffs in sorted(layer_dict.items())
    ])                                   # shape: (18, 4)
    for example_idx, layer_dict in sorted(attn_corrupt_logits.items())
])                                       # shape: (100, 18, 4)

print(all_attn_diffs.shape)

torch.Size([100, 18, 4])


In [64]:
mean_per_layer_head = all_attn_diffs.mean(dim=0)  # shape: (18, 4)
print(mean_per_layer_head)
for layer_idx in range(18):
    for head_idx in range(4):
        print(f"layer {layer_idx} head {head_idx}: {mean_per_layer_head[layer_idx][head_idx]:.4f}")

tensor([[5.0625, 4.8438, 5.0625, 5.0625],
        [5.0625, 5.0625, 5.0625, 5.0625],
        [5.0312, 5.0938, 5.0625, 5.0625],
        [5.0938, 5.0625, 5.0625, 4.7188],
        [5.1250, 5.1250, 5.0625, 5.0312],
        [5.0312, 4.9375, 3.5312, 4.4688],
        [5.0625, 4.9375, 5.0625, 5.1250],
        [4.8125, 5.0000, 5.0000, 5.0625],
        [5.0312, 5.0625, 5.0000, 5.0000],
        [5.0000, 5.4062, 5.0625, 4.3125],
        [5.0938, 5.0625, 5.1250, 4.2812],
        [5.0938, 4.5000, 6.0000, 3.0000],
        [5.0625, 4.9062, 5.3125, 5.2812],
        [4.9062, 5.0625, 5.0312, 4.7500],
        [5.0625, 5.0625, 5.0625, 5.1250],
        [4.5625, 6.9688, 4.2812, 4.2812],
        [5.0625, 5.0312, 5.0312, 5.0625],
        [4.8125, 5.0000, 6.1250, 5.0625]], dtype=torch.bfloat16)
layer 0 head 0: 5.0625
layer 0 head 1: 4.8438
layer 0 head 2: 5.0625
layer 0 head 3: 5.0625
layer 1 head 0: 5.0625
layer 1 head 1: 5.0625
layer 1 head 2: 5.0625
layer 1 head 3: 5.0625
layer 2 head 0: 5.0312
layer 2 head 1

In [65]:
mean_attn_diff = clean_prompt_mean - mean_per_layer_head  #to see the deviation from clean prompt
mean_attn_diff

tensor([[ 0.0000,  0.2188,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0312, -0.0312,  0.0000,  0.0000],
        [-0.0312,  0.0000,  0.0000,  0.3438],
        [-0.0625, -0.0625,  0.0000,  0.0312],
        [ 0.0312,  0.1250,  1.5312,  0.5938],
        [ 0.0000,  0.1250,  0.0000, -0.0625],
        [ 0.2500,  0.0625,  0.0625,  0.0000],
        [ 0.0312,  0.0000,  0.0625,  0.0625],
        [ 0.0625, -0.3438,  0.0000,  0.7500],
        [-0.0312,  0.0000, -0.0625,  0.7812],
        [-0.0312,  0.5625, -0.9375,  2.0625],
        [ 0.0000,  0.1562, -0.2500, -0.2188],
        [ 0.1562,  0.0000,  0.0312,  0.3125],
        [ 0.0000,  0.0000,  0.0000, -0.0625],
        [ 0.5000, -1.9062,  0.7812,  0.7812],
        [ 0.0000,  0.0312,  0.0312,  0.0000],
        [ 0.2500,  0.0625, -1.0625,  0.0000]], dtype=torch.bfloat16)

In [ ]:
scores_flat = [
    (layer_idx, head_idx, mean_attn_diff[layer_idx][head_idx].item())
    for layer_idx in range(18)
    for head_idx in range(4)
]
ranked = sorted(scores_flat, key=lambda x: x[2], reverse=True)
ranked

In [69]:
selected_attn_components = [(layer, head, r) for layer,head, r in ranked if r>threshold]
selected_attn_components

[(11, 3, 2.0625),
 (5, 2, 1.53125),
 (10, 3, 0.78125),
 (15, 2, 0.78125),
 (15, 3, 0.78125),
 (9, 3, 0.75),
 (5, 3, 0.59375),
 (11, 1, 0.5625),
 (15, 0, 0.5),
 (3, 3, 0.34375),
 (13, 3, 0.3125)]

### Checking sufficiency
corrupt other clean selected

In [99]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [100]:
def mlp_hook(name, corrupt):
    def hook(model, input, output): 
        output = corrupt.to(device=device)
        return output
    return hook

In [101]:
def attn_hook(name, corrupt, e):
    def hook(module, input, output):
    
        for j in range(4):
            patched_input = input[0].clone()
            patched_input[:, :, j*256:(j+1)*256] = torch.tensor(
                corrupt[name][str(j)][e], 
                dtype=patched_input.dtype, 
                device=patched_input.device
            )
            
            with torch.no_grad():
                patched_output = torch.nn.functional.linear(
                    patched_input, module.weight, module.bias
                )
        return patched_output
    return hook

In [86]:
print(selected_mlp_components)
mlp_layers = [l for l, act in selected_mlp_components]
mlp_layers

[(3, 4.3125), (10, 1.078125), (11, 0.90625), (9, 0.84375), (6, 0.65625), (7, 0.46875), (12, 0.4375), (8, 0.375)]


[3, 10, 11, 9, 6, 7, 12, 8]

In [87]:
print(selected_attn_components)
attn_layer = [l for l,h,act in selected_attn_components]
attn_head = [h for l,h,act in selected_attn_components]
print(attn_layer)
print(attn_head)

[(11, 3, 2.0625), (5, 2, 1.53125), (10, 3, 0.78125), (15, 2, 0.78125), (15, 3, 0.78125), (9, 3, 0.75), (5, 3, 0.59375), (11, 1, 0.5625), (15, 0, 0.5), (3, 3, 0.34375), (13, 3, 0.3125)]
[11, 5, 10, 15, 15, 9, 5, 11, 15, 3, 13]
[3, 2, 3, 2, 3, 3, 3, 1, 0, 3, 3]


In [102]:
import gc
torch.cuda.empty_cache()
gc.collect()

22336

In [103]:
suff_logit = []

for e, example in enumerate(data):
    h = []
    for i in range(18):
        for name, layer in  model.named_modules():
            if name==f'model.layers.{i}.mlp' and i not in mlp_layers:
                h.append(layer.register_forward_hook(mlp_hook(name, torch.tensor(corrupt_activations[name][e], dtype=torch.bfloat16, device=device))))
            if name==f'model.layers.{i}.self_attn.o_proj' and i not in attn_layer:
                h.append(layer.register_forward_hook(attn_hook(name, corrupt_activations, e)))
    tokenize = tokenizer(example['clean_prompt'], return_tensors='pt').to(device = device)
    with torch.no_grad():
        output = model(**tokenize)
    A_token = tokenizer("▁"+example["name_A"], return_tensors='pt').to(device = device)
    B_token = tokenizer("▁"+example["name_B"], return_tensors='pt').to(device = device)
    A_id = A_token["input_ids"][0][-1] #extracting token id of A
    B_id = B_token["input_ids"][0][-1]
    A_logit = output.logits[:,-1,:][0][int(A_id)] #taking out logit of A from the last token's logit
    B_logit = output.logits[:,-1,:][0][int(B_id)]
    suff_logit.append((A_logit - B_logit).detach().cpu())
    for k in h:
        k.remove()
    

In [ ]:
print(len(suff_logit))
suff_logit

In [109]:
clean_prompt_mean

tensor(5.0625, dtype=torch.bfloat16)

In [120]:
suff_mean = torch.tensor(suff_logit).mean()
suff_mean

tensor(4.1250, dtype=torch.bfloat16)

In [112]:
clean_prompt_mean - suff_mean

tensor(0.9375, dtype=torch.bfloat16)

### checking necessity
corrupt the selected

In [113]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [121]:
import gc
torch.cuda.empty_cache()
gc.collect()

0

In [115]:
ness_logit = []

for e, example in enumerate(data):
    h = []
    for i in range(18):
        for name, layer in  model.named_modules():
            if name==f'model.layers.{i}.mlp' and i in mlp_layers:
                h.append(layer.register_forward_hook(mlp_hook(name, torch.tensor(corrupt_activations[name][e], dtype=torch.bfloat16, device=device))))
            if name==f'model.layers.{i}.self_attn.o_proj' and i in attn_layer:
                h.append(layer.register_forward_hook(attn_hook(name, corrupt_activations, e)))
    tokenize = tokenizer(example['clean_prompt'], return_tensors='pt').to(device = device)
    with torch.no_grad():
        output = model(**tokenize)
    A_token = tokenizer("▁"+example["name_A"], return_tensors='pt').to(device = device)
    B_token = tokenizer("▁"+example["name_B"], return_tensors='pt').to(device = device)
    A_id = A_token["input_ids"][0][-1] #extracting token id of A
    B_id = B_token["input_ids"][0][-1]
    A_logit = output.logits[:,-1,:][0][int(A_id)] #taking out logit of A from the last token's logit
    B_logit = output.logits[:,-1,:][0][int(B_id)]
    ness_logit.append((A_logit - B_logit).detach().cpu())
    for k in h:
        k.remove()

In [ ]:
ness_logit

In [118]:
ness_mean = torch.tensor(ness_logit).mean()
ness_mean

tensor(0.5430, dtype=torch.bfloat16)